### Links and Resources
| Concept | Reference |
|---|---|
| [`pyspark.sql.functions.udf`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udf.html) | The UDF registration helper |
| [Python UDF programming guide](https://spark.apache.org/docs/latest/api/python/user_guide/sql/python_udf.html) | Official guide |
| [Arrow-optimized Python UDFs](https://www.databricks.com/blog/arrow-optimized-python-udfs-apache-sparktm-35) | The 3.5+ speedup |
| [Spark SQL types](https://spark.apache.org/docs/latest/sql-ref-datatypes.html) | Return type strings |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# Python UDFs

A **User-Defined Function** lets you wrap arbitrary Python in something that behaves like a column expression. The mechanics are simple; the cost is not.

This notebook covers:

1. The two registration styles (decorator and `udf()` call).
2. Why a plain Python UDF is **5–50× slower** than the equivalent built-in.
3. NULL handling and type pitfalls.
4. The Arrow-optimized variant (Spark 3.5+) that closes most of the gap.

## Why UDFs are slow

For each row, Spark must:

1. Serialize the row from JVM → bytes (via Pickle).
2. Send it to a Python worker process.
3. Deserialize, run your function, serialize the result.
4. Send bytes back to the JVM and deserialize.

That's the overhead — round-trip per row, no Catalyst optimization, no Photon. **Built-ins skip all of it.** The first rule of UDFs: *check the built-in catalog first.*

In [ ]:
# Demo data — fake "names" we want to clean up
import random
random.seed(0)
N = 50_000
words = ["alice", "BOB", "  charlie ", "dana", "ERIN", "frank", "  grace"]
df = spark.createDataFrame(
    [(i, random.choice(words)) for i in range(N)],
    ["id", "name"],
)
df.show(5, truncate=False)
print("rows:", df.count())

## Style 1 — register with `udf()` and a return type

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def normalize(s):
    if s is None:
        return None
    return s.strip().title()

normalize_udf = udf(normalize, StringType())

df.select("name", normalize_udf("name").alias("clean")).show(5, truncate=False)

## Style 2 — decorator with a return-type string

Equivalent, often nicer to read.

In [ ]:
@udf("string")
def normalize2(s):
    return s.strip().title() if s is not None else None

df.select("name", normalize2("name").alias("clean")).show(5, truncate=False)

## NULL handling — your function will receive `None`

Spark calls your UDF for every row, including those with NULL inputs. If you forget the guard, a NULL crashes the whole task with `AttributeError: 'NoneType' object has no attribute ...`.

In [ ]:
@udf("string")
def safe_upper(s):
    return None if s is None else s.upper()

with_nulls = df.union(spark.createDataFrame([(99999, None)], ["id", "name"]))
with_nulls.select(safe_upper("name").alias("upper")).filter(F.col("upper").isNull()).show(2)

## Return types — be explicit, be correct

The return-type string (or `DataType` object) is **not optional**. Spark uses it to allocate the output column and decide what JVM type to convert to. Common ones:

| Python value | Spark type string |
|---|---|
| `int` | `"int"` (32-bit) or `"long"` (64-bit) |
| `float` | `"double"` |
| `str` | `"string"` |
| `bool` | `"boolean"` |
| `list[int]` | `"array<int>"` |
| `dict` | `"map<string,string>"` |
| `tuple` / `dataclass` | `"struct<a:int,b:string>"` |

Mismatches don't always raise — sometimes you just get NULL columns. Test before you ship.

In [ ]:
@udf("array<string>")
def split_words(s):
    if s is None:
        return None
    return [w for w in s.strip().split() if w]

spark.createDataFrame([("hello world",), ("a b c",), (None,)], ["t"]) \
    .select("t", split_words("t").alias("words")).show(truncate=False)

## A struct-returning UDF

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

@udf(StructType([
    StructField("clean",  StringType()),
    StructField("length", IntegerType()),
]))
def parse(s):
    if s is None:
        return None
    cleaned = s.strip().title()
    return (cleaned, len(cleaned))

df.select(parse("name").alias("p")).select("p.*").show(5)

## Using a UDF in SQL — `spark.udf.register`

Once registered with a name, it works inside any SQL string.

In [ ]:
spark.udf.register("normalize_sql", normalize, "string")

df.createOrReplaceTempView("v")
spark.sql("SELECT id, normalize_sql(name) AS clean FROM v LIMIT 5").show(truncate=False)

## Arrow-optimized Python UDFs (Spark 3.5+ / DBR 14+)

The `useArrow=True` flag swaps the row-by-row Pickle pipe for Arrow batches — same Python function, much less serialization overhead. Often **2–5× faster** than the classic UDF, with no code changes.

In [ ]:
@udf("string", useArrow=True)
def normalize_arrow(s):
    return s.strip().title() if s is not None else None

# Same call site as before
df.select(normalize_arrow("name").alias("clean")).show(5, truncate=False)

## Quick benchmark — built-in vs UDF vs Arrow UDF

We expect: **built-in fastest by a wide margin**, Arrow UDF middle, classic UDF slowest. The exact ratio varies by cluster and Spark version.

In [ ]:
import time

def bench(label, df_expr):
    df_expr.write.format("noop").mode("overwrite").save()  # forces full computation, no I/O
    t = time.time()
    df_expr.write.format("noop").mode("overwrite").save()
    print(f"  {label:30s} {time.time() - t:5.2f}s")

big = df.union(df).union(df).union(df)            # ~200k rows
print("rows:", big.count())

bench("built-in",       big.select(F.initcap(F.trim("name"))))
bench("classic UDF",    big.select(normalize2("name")))
bench("Arrow UDF",      big.select(normalize_arrow("name")))

## When a UDF *is* the right answer

Don't over-rotate against UDFs. Reach for one when:

- The logic genuinely needs a Python library (e.g. `phonenumbers`, `dateutil`, an internal package).
- The transformation is complex enough that expressing it in built-ins becomes unreadable.
- The data volume is small (< few million rows) and developer time matters more than runtime.

In those cases, **prefer Arrow UDFs**, and **prefer Pandas UDFs** (next notebook) when you can vectorize naturally.

## Takeaways

- A Python UDF is the slowest way to express a transformation; check the built-in catalog first.
- Always set the return type explicitly. Always handle `None`.
- Register with `spark.udf.register("name", fn, type)` to use it in SQL.
- `useArrow=True` (Spark 3.5+) gives a free 2–5× speedup over the classic UDF.
- Save UDFs for genuine library-dependent logic; vectorize via Pandas UDFs (notebook 12) when possible.